Working Title:

Expected Goals (xG) - a better statistic than simply counting shots?


Unit of Analysis
Run at both levels for robustness, but use match-level as primary:

Level	N	Notes
Team-match	~3,800	380 matches × 5 seasons × 2 teams per match
Team-season	~100	Aggregated; noise-reduced but low power
Each row at match level = one team's stats in one match. This gives you enough power to detect meaningful differences.

How this question came up:
- Soccer is a high-variance, low-scoring sport. You can be a better team on paper and still win far less than you'd expect.
- e.g. Chelsea 2015-2016 with 37.74% win rate.


Take data from last 5 years.
Compute:
1. Total number of shots per season by a team
2. 

## Test Plan

### Hypotheses
- **H₀**: shots/match and xG/match are equally good predictors of goals/match
- **H₁**: xG/match is a better predictor of goals/match than shots/match

---

### Unit of Analysis
- **Team-match** (primary): each row = one team's stats in one match
- ~3,800 observations (380 matches × 5 seasons × 2 teams per match)

---

### Variables

| Variable | Definition | Role |
|---|---|---|
| `goals` | Goals scored by team in match | Outcome (DV) |
| `shots` | Total shots taken | Quantity predictor |
| `xG` | Expected goals generated | Quality + volume predictor |
| `xG_per_shot` | xG / shots | Pure quality (volume-neutral) |

---

### Step 1 — Descriptive: Correlation comparison (no test)

Compute and report Pearson and Spearman correlations side-by-side:

| | Pearson r | Spearman ρ |
|---|---|---|
| shots vs goals | r₁ | ρ₁ |
| xG vs goals | r₂ | ρ₂ |

If r₂ > r₁ and ρ₂ > ρ₁, both metrics agree directionally. No formal test here — this is description only.

---

### Step 2 — Linear regression R² comparison

Fit two OLS models (include `home` dummy and season dummies as controls in both):

```
Model A: goals ~ shots        → R²_A
Model B: goals ~ xG           → R²_B
```

ΔR² = R²_B − R²_A is the effect size. Still descriptive — Step 3 tests whether the gap is statistically real.

---

### Step 3 — Formal test: Paired t-test on absolute prediction errors

This is the significance test for H₀.

For each match observation, compute the absolute prediction error from each model:
- e_A = |goals − ŷ_A| (from Model A)
- e_B = |goals − ŷ_B| (from Model B)
- d = e_A − e_B (positive d means xG made a smaller error on that match)

Run a one-sided paired t-test on d against a population mean of 0:
- H₀: mean(d) = 0 (both models err equally)
- H₁: mean(d) > 0 (Model B / xG makes smaller errors)

CLT covers the normality assumption at N ≈ 3,800.

---

### Step 4 — Does shots add anything beyond xG?

Fit a joint regression with standardised predictors (so coefficients are directly comparable):

```
Model C: goals ~ shots_z + xG_z
```

Read the t-test on β_shots directly from the regression output:
- H₀: β_shots = 0 (shots contributes nothing beyond xG)
- If we fail to reject H₀, xG fully subsumes shot counts

---

### Decision Logic

```
Step 3 significant (p < 0.05)?
├── No  → "xG is not a statistically better predictor than shots"
└── Yes → Step 4: β_shots significant?
          ├── Yes → "xG is better, but shot volume still carries independent signal"
          └── No  → "xG fully subsumes shot counts — it is the superior statistic"
```